# 01 — Collect a Training Dataset

This is the first of three notebooks that build a working MOUSE agent end-to-end:

| Notebook | What it does |
|---|---|
| **01 — Collect** (this one) | Run two FrozenLake environments with random actions and save the transitions to the Hub |
| **02 — Train** | Load those transitions and train a small DQN model offline |
| **03 — Inference** | Run the trained model live and watch it play |

MOUSE organises data into named `Datastore`s — one per environment — which map directly to Hugging Face dataset configs. That makes loading a specific environment's data by name straightforward in the training notebook.


In [9]:
from mouse_envs import EnvConfig, make_env
from mouse_core.data import Datastore, push_stores_to_hub

DATASET_ID = "mouse-example-dataset"  # created under your authenticated HF user
STEPS_PER_ENV = 200                   # random transitions to collect per environment


## Datastores

A `Datastore` is a sequential container for step data backed by a Hugging Face `Dataset` (Arrow). It stores rows — plain Python dicts — in the exact order they arrive. There is no required schema: each row contains whatever your collection script produced.

- `store.append(row)` adds a single step during collection.
- `store.append(other_store)` / `store.append([stores])` merges from another store.
- `Datastore.from_dataset(hf_dataset)` wraps an existing HF dataset.

The `name` given to a store becomes its config identifier on the Hub, which is how `load_stores_from_hub` retrieves it by name in the training notebook.

## Environments

We define two FrozenLake variants — one slippery (stochastic transitions) and one deterministic — and pass them together to `make_env`. Each config's `name` becomes the `Datastore` identifier so the training notebook can load each variant individually.


In [10]:
# FrozenLake has four discrete actions, matching the DQN head in the training notebook.
configs = [
    EnvConfig("FrozenLake-v1", name="frozenlake_slippery",     seed=0, num_envs=2, episodes_per_task=5, kwargs={"is_slippery": True}),
    EnvConfig("FrozenLake-v1", name="frozenlake_deterministic", seed=0, num_envs=1, episodes_per_task=5, kwargs={"is_slippery": False}),
]
env = make_env(configs)

## Collect transitions

At each step we sample a random action for every inner environment simultaneously, apply it, and save the `(action, observation, reward, done)` result to each environment's `Datastore`. The `Datastore` name identifies which environment produced each row.

In [11]:
def collect_steps(stores, env, num_steps):
    for _ in range(num_steps):
        inputs = env.sample_random_inputs()
        outputs = env.step(inputs)
        for store, inp, out in zip(stores, inputs, outputs):
            store.append({**inp, **out})


stores = [Datastore(name=name) for name in env.names]
collect_steps(stores, env, STEPS_PER_ENV)
env.close()

for store in stores:
    print(store)

Datastore(name='frozenlake_slippery#0', steps=200)
Datastore(name='frozenlake_slippery#1', steps=200)
Datastore(name='frozenlake_deterministic#0', steps=200)


## Push to the Hub

Each `Datastore` is uploaded as a separate named config inside the dataset repo. Anyone with the repo ID can call `load_stores_from_hub` to retrieve them by name.


In [12]:
url = push_stores_to_hub(stores, repo_id=DATASET_ID, split="train", private=False, clear=True)
print(f"Pushed to {url}")


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to micahr234/mouse-example-dataset (frozenlake_slippery#0/train: 200, frozenlake_slippery#1/train: 200, frozenlake_deterministic#0/train: 200 steps)
Pushed to https://huggingface.co/datasets/micahr234/mouse-example-dataset


## Hub structure

Each store is saved as a named **config** (also called subset) inside a single dataset repository. Configs are the natural way to organise different "bins" of data — different environments, policies, seeds — inside one repo.

Load all configs from a repo:

```python
from mouse_core.data import load_stores_from_hub

stores = load_stores_from_hub("your-dataset", split="train")
```

Or load only specific configs by name:

```python
stores = load_stores_from_hub("your-dataset", ["frozenlake_slippery"], split="train")
```

For custom loading workflows you can go through the HF `Dataset` directly and wrap the result in a store:

```python
from datasets import load_dataset
from mouse_core.data import Datastore

ds = load_dataset("your-org/your-dataset", name="frozenlake_slippery", split="train")
store = Datastore.from_dataset(ds)
```

Pushing through `push_stores_to_hub` deletes the previous README and parquet shards before uploading so the dataset card always reflects the current data. To organise configs and splits declaratively in the repo README, add a YAML block following the [Hugging Face repository structure guide](https://huggingface.co/docs/datasets/repository_structure#define-your-splits-and-subsets-in-yaml):

```yaml
configs:
- config_name: frozenlake_slippery
  data_files:
  - split: train
    path: "data/frozenlake_slippery/train/*.parquet"
- config_name: frozenlake_deterministic
  data_files:
  - split: train
    path: "data/frozenlake_deterministic/train/*.parquet"
```